# GeoGuessr Challenge - Cached-Embedding Multi-Task Pipeline

Compliance summary (see rules 4.1-4.4):
- Backbones: `openai/clip-vit-large-patch14` (image-text contrastive) and `facebook/dinov2-large`
  (self-supervised). Neither was trained to predict where an image was taken. No StreetCLIP,
  no GeoCLIP, no osv5m/baseline, no embeddings or distillation from any geolocation model.
- All backbone weights are downloaded ONCE and saved to disk. Inference reloads them from the
  local path with HF offline flags set, and Cell 14 proves this works with the hub disabled.
- No language-generation head anywhere. Only frozen visual encoders + heads trained from scratch.
- Free-tier Kaggle T4. Everything is seeded.

Run order: just Run All. Expect roughly 2-4 hours depending on external-data settings.

In [1]:
# =====================================================================
# CELL 1 - Installs
# =====================================================================
!pip install -q transformers==4.44.2 huggingface_hub shapely scikit-learn pandas numpy pillow tqdm openpyxl
print("installs done", flush=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 80.5 MB/s eta 0:00:00
installs done


In [2]:
# =====================================================================
# CELL 2 - Imports, config, logging
# =====================================================================
import os, sys, gc, io, json, math, time, glob, random, zipfile, shutil, warnings, traceback
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
from tqdm.auto import tqdm
from sklearn.cluster import MiniBatchKMeans

T0 = time.time()
def log(msg):
    el = time.time() - T0
    print(f"[{time.strftime('%H:%M:%S')} | +{el/60:6.1f} min] {msg}", flush=True)

class CFG:
    SEED               = 42
    WORK               = "/kaggle/working"
    ART                = "/kaggle/working/artifacts"
    TMP                = "/kaggle/temp"

    # --- backbones (neither is geolocation-trained) ---
    CLIP_ID            = "openai/clip-vit-large-patch14"
    DINO_ID            = "facebook/dinov2-large"
    USE_DINO           = False         # CLIP only: buys data breadth instead
    IMG                = 224
    EXTRACT_BS         = 48
    WORKERS            = 4

    # --- external data (OSV-5M is a DATASET, permitted under sec 3.2) ---
    USE_EXTERNAL       = True
    EXT_REPO           = "osv5m/osv5m"
    EXT_SHARDS         = 5
    MAX_EXTERNAL       = 250_000

    # --- geocells / heads ---
    N_FINE             = 2000
    N_COARSE           = 150
    TAU_KM             = 250.0
    TOP_K              = 8

    # --- head training ---
    N_FOLDS            = 5
    EPOCHS             = 60
    HEAD_BS            = 2048
    LR                 = 2e-3
    WD                 = 1e-4
    HID                = 1024
    W_PROVIDED         = 3.0     # in-domain images weigh more than external
    W_EXTERNAL         = 1.0
    Q_UNC              = 0.70    # pinball quantile for the uncertainty head

    # --- radius policy search ---
    R_MIN              = 15.0
    R_MAX              = 3000.0

    # --- stage 2: backbone fine-tuning (paper-validated lever) ---
    RUN_FINETUNE       = True
    FT_UNFREEZE_BLOCKS = 4
    FT_EPOCHS          = 3
    FT_BS              = 32
    FT_LR              = 8e-6      # backbone
    FT_HEAD_LR         = 4e-4      # head
    FT_MAX_IMAGES      = 150_000
    FT_MAX_MINUTES     = 200       # hard wall-clock guard

R_EARTH = 6371.0088

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_all(CFG.SEED)

os.makedirs(CFG.ART, exist_ok=True)
os.makedirs(CFG.TMP, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log(f"device={device} | torch={torch.__version__}")
if torch.cuda.is_available():
    log(f"gpu={torch.cuda.get_device_name(0)}")

# ---- geometry helpers (vectorised; no python double loops anywhere) ----
def latlon_to_vec(lat, lon):
    la = np.radians(np.asarray(lat, dtype=np.float64))
    lo = np.radians(np.asarray(lon, dtype=np.float64))
    return np.stack([np.cos(la)*np.cos(lo), np.cos(la)*np.sin(lo), np.sin(la)], axis=-1)

def vec_to_latlon(v):
    v = np.asarray(v, dtype=np.float64)
    v = v / (np.linalg.norm(v, axis=-1, keepdims=True) + 1e-12)
    lat = np.degrees(np.arcsin(np.clip(v[..., 2], -1, 1)))
    lon = np.degrees(np.arctan2(v[..., 1], v[..., 0]))
    return lat, lon

def hav_km(lat1, lon1, lat2, lon2):
    la1, lo1, la2, lo2 = map(lambda x: np.radians(np.asarray(x, dtype=np.float64)),
                             (lat1, lon1, lat2, lon2))
    d = np.sin((la2-la1)/2)**2 + np.cos(la1)*np.cos(la2)*np.sin((lo2-lo1)/2)**2
    return 2*R_EARTH*np.arcsin(np.sqrt(np.clip(d, 0, 1)))

log("config ready")

[16:43:11 | +   0.0 min] device=cuda | torch=2.10.0+cu128
[16:43:11 | +   0.0 min] gpu=Tesla T4
[16:43:11 | +   0.0 min] config ready


In [3]:
# =====================================================================
# CELL 3 - Auto-discover competition files (no hardcoded paths)
# =====================================================================
def find_all(pattern, root="/kaggle/input"):
    return sorted(glob.glob(os.path.join(root, "**", pattern), recursive=True))

SAMPLE_SUB = find_all("sample*submission*.csv")
GEOJSON    = find_all("*.geojson")
GT_FILES   = [p for p in find_all("*.csv") + find_all("*.xlsx")
              if "ground" in os.path.basename(p).lower()
              or "coordinate" in os.path.basename(p).lower()]

assert SAMPLE_SUB, "sample_submission.csv not found - attach the competition data via Add Input"
SAMPLE_SUB = SAMPLE_SUB[0]
GEOJSON    = GEOJSON[0] if GEOJSON else None
log(f"sample_submission : {SAMPLE_SUB}")
log(f"country geojson   : {GEOJSON}")

sub_template = pd.read_csv(SAMPLE_SUB)
log(f"submission columns (authoritative): {list(sub_template.columns)}")

def pick_col(cols, *keys):
    for c in cols:
        lc = c.lower().replace("_", "").replace(" ", "")
        if all(k in lc for k in keys):
            return c
    return None

SUB_ID  = pick_col(sub_template.columns, "id") or sub_template.columns[0]
SUB_LAT = pick_col(sub_template.columns, "lat")
SUB_LON = pick_col(sub_template.columns, "lon") or pick_col(sub_template.columns, "lng")
SUB_RAD = pick_col(sub_template.columns, "rad")
assert None not in (SUB_LAT, SUB_LON, SUB_RAD), f"could not map columns: {list(sub_template.columns)}"
log(f"mapped -> id={SUB_ID} lat={SUB_LAT} lon={SUB_LON} radius={SUB_RAD}")

# ---- training ground truth ----
assert GT_FILES, "ground truth file not found"
GT_PATH = GT_FILES[0]
gt = pd.read_excel(GT_PATH) if GT_PATH.endswith(".xlsx") else pd.read_csv(GT_PATH)
log(f"ground truth      : {GT_PATH}  shape={gt.shape}")
log(f"gt columns        : {list(gt.columns)}")

GT_LAT = pick_col(gt.columns, "lat")
GT_LON = pick_col(gt.columns, "lon") or pick_col(gt.columns, "lng")
GT_ID  = pick_col(gt.columns, "id") or pick_col(gt.columns, "image") or pick_col(gt.columns, "file") or gt.columns[0]
assert None not in (GT_LAT, GT_LON), "lat/lon columns not found in ground truth"
log(f"gt mapped -> id={GT_ID} lat={GT_LAT} lon={GT_LON}")

# ---- image folders ----
ALL_IMGS = find_all("*.jpg") + find_all("*.jpeg") + find_all("*.png")
log(f"images visible under /kaggle/input: {len(ALL_IMGS)}")

test_ids = sub_template[SUB_ID].astype(str).tolist()
test_stem_set = set(os.path.splitext(t)[0] for t in test_ids)

by_stem = {}
for p in ALL_IMGS:
    by_stem.setdefault(os.path.splitext(os.path.basename(p))[0], p)

TEST_PATHS = []
missing = 0
for t in test_ids:
    s = os.path.splitext(str(t))[0]
    if s in by_stem:
        TEST_PATHS.append(by_stem[s])
    else:
        TEST_PATHS.append(None); missing += 1
log(f"test images matched: {len(TEST_PATHS)-missing}/{len(TEST_PATHS)}  (missing={missing})")

gt["stem"] = gt[GT_ID].astype(str).map(lambda x: os.path.splitext(str(x))[0])
gt["path"] = gt["stem"].map(by_stem.get)
before = len(gt)
gt = gt[gt["path"].notna()].copy()
# never train on anything that is also a test image
gt = gt[~gt["stem"].isin(test_stem_set)].copy()
log(f"train rows with resolvable images: {len(gt)}/{before}")

train_df = pd.DataFrame({
    "path": gt["path"].values,
    "lat":  pd.to_numeric(gt[GT_LAT], errors="coerce").values,
    "lon":  pd.to_numeric(gt[GT_LON], errors="coerce").values,
    "src":  "provided",
})
train_df = train_df.dropna(subset=["lat", "lon"])
train_df = train_df[(train_df.lat.between(-90, 90)) & (train_df.lon.between(-180, 180))]
train_df = train_df.reset_index(drop=True)
log(f"provided training rows: {len(train_df)}")

[16:43:50 | +   0.7 min] sample_submission : /kaggle/input/competitions/geolocation-prediction/sample_submission.csv
[16:43:50 | +   0.7 min] country geojson   : /kaggle/input/competitions/geolocation-prediction/country_boundaries.geojson
[16:43:50 | +   0.7 min] submission columns (authoritative): ['image_id', 'pred_lat', 'pred_lon', 'pred_radius_km']
[16:43:50 | +   0.7 min] mapped -> id=image_id lat=pred_lat lon=pred_lon radius=pred_radius_km
[16:43:50 | +   0.7 min] ground truth      : /kaggle/input/competitions/geolocation-prediction/training_dataset/noised_dataset/ground_truth_coordinates.csv  shape=(19002, 3)
[16:43:50 | +   0.7 min] gt columns        : ['image_id', 'latitude', 'longitude']
[16:43:50 | +   0.7 min] gt mapped -> id=image_id lat=latitude lon=longitude
[16:43:50 | +   0.7 min] images visible under /kaggle/input: 19502
[16:43:50 | +   0.7 min] test images matched: 500/500  (missing=0)
[16:43:50 | +   0.7 min] train rows with resolvable images: 19002/19002
[16:43:50 

In [4]:
# =====================================================================
# CELL 4 - Download backbones ONCE and save them locally (rule 4.3)
# =====================================================================
from transformers import CLIPVisionModel, AutoModel

CLIP_LOCAL = os.path.join(CFG.ART, "clip_vit_l14")
DINO_LOCAL = os.path.join(CFG.ART, "dinov2_large")

if not os.path.exists(os.path.join(CLIP_LOCAL, "config.json")):
    log("downloading CLIP vision tower (internet required, TRAINING ONLY)...")
    m = CLIPVisionModel.from_pretrained(CFG.CLIP_ID)
    m.save_pretrained(CLIP_LOCAL)
    del m; gc.collect()
log(f"CLIP saved locally -> {CLIP_LOCAL}")

if CFG.USE_DINO and not os.path.exists(os.path.join(DINO_LOCAL, "config.json")):
    try:
        log("downloading DINOv2-large...")
        m = AutoModel.from_pretrained(CFG.DINO_ID)
        m.save_pretrained(DINO_LOCAL)
        del m; gc.collect()
    except Exception as e:
        log(f"DINOv2 unavailable ({e}) - continuing with CLIP only")
        CFG.USE_DINO = False
if CFG.USE_DINO:
    log(f"DINOv2 saved locally -> {DINO_LOCAL}")

# from here on, everything loads from disk
BACKBONES = {}
clip_m = CLIPVisionModel.from_pretrained(CLIP_LOCAL).to(device).half().eval()
for p in clip_m.parameters(): p.requires_grad = False
BACKBONES["clip"] = dict(
    model=clip_m,
    mean=torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1,3,1,1).half(),
    std =torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1,3,1,1).half(),
    dim =clip_m.config.hidden_size, kind="clip")

if CFG.USE_DINO:
    dino_m = AutoModel.from_pretrained(DINO_LOCAL).to(device).half().eval()
    for p in dino_m.parameters(): p.requires_grad = False
    BACKBONES["dino"] = dict(
        model=dino_m,
        mean=torch.tensor([0.485, 0.456, 0.406], device=device).view(1,3,1,1).half(),
        std =torch.tensor([0.229, 0.224, 0.225], device=device).view(1,3,1,1).half(),
        dim =dino_m.config.hidden_size, kind="dino")

N_VIEWS = 2
FEAT_DIM = sum(b["dim"] for b in BACKBONES.values()) * N_VIEWS
log(f"backbones={list(BACKBONES)} | feature dim = {FEAT_DIM}")

[16:43:51 | +   0.7 min] downloading CLIP vision tower (internet required, TRAINING ONLY)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

[16:44:07 | +   0.9 min] CLIP saved locally -> /kaggle/working/artifacts/clip_vit_l14
[16:44:08 | +   1.0 min] backbones=['clip'] | feature dim = 2048


In [5]:
# =====================================================================
# CELL 5 - External data (OSV-5M is a dataset, not a model - sec 3.2)
# =====================================================================
ext_df = pd.DataFrame(columns=["path", "lat", "lon", "src"])

if CFG.USE_EXTERNAL:
    try:
        from huggingface_hub import list_repo_files, hf_hub_download
        files = list_repo_files(CFG.EXT_REPO, repo_type="dataset")
        zips  = sorted([f for f in files if f.endswith(".zip") and "train" in f.lower()])
        metas = sorted([f for f in files if f.endswith(".csv") and "train" in f.lower()])
        log(f"OSV-5M: {len(zips)} train zips, meta candidates={metas[:3]}")
        assert zips and metas

        meta_path = hf_hub_download(CFG.EXT_REPO, metas[0], repo_type="dataset",
                                    local_dir=CFG.TMP)
        head = pd.read_csv(meta_path, nrows=5)
        m_lat = pick_col(head.columns, "lat"); m_lon = pick_col(head.columns, "lon")
        m_id  = pick_col(head.columns, "id") or head.columns[0]
        log(f"OSV-5M meta cols -> id={m_id} lat={m_lat} lon={m_lon}")
        meta = pd.read_csv(meta_path, usecols=[m_id, m_lat, m_lon])
        meta.columns = ["ext_id", "lat", "lon"]
        meta["ext_id"] = meta["ext_id"].astype(str)
        log(f"OSV-5M meta rows: {len(meta)}")

        ext_dir = os.path.join(CFG.TMP, "osv5m_imgs")
        os.makedirs(ext_dir, exist_ok=True)
        for z in zips[:CFG.EXT_SHARDS]:
            log(f"downloading shard {z} ...")
            zp = hf_hub_download(CFG.EXT_REPO, z, repo_type="dataset", local_dir=CFG.TMP)
            with zipfile.ZipFile(zp) as zf:
                zf.extractall(ext_dir)
            os.remove(zp)
            log(f"extracted {z}")

        ext_paths = glob.glob(os.path.join(ext_dir, "**", "*.jpg"), recursive=True)
        ext_paths = sorted(ext_paths)                      # sorted -> reproducible
        log(f"external images on disk: {len(ext_paths)}")
        emap = pd.DataFrame({
            "path": ext_paths,
            "ext_id": [os.path.splitext(os.path.basename(p))[0] for p in ext_paths]})
        emap = emap[~emap.ext_id.isin(test_stem_set)]      # contamination guard
        ext_df = emap.merge(meta, on="ext_id", how="inner")
        ext_df = ext_df.dropna(subset=["lat", "lon"])
        if len(ext_df) > CFG.MAX_EXTERNAL:
            ext_df = ext_df.sample(CFG.MAX_EXTERNAL, random_state=CFG.SEED)
        ext_df = ext_df[["path", "lat", "lon"]].copy()
        ext_df["src"] = "external"
        log(f"external rows kept: {len(ext_df)}")
    except Exception as e:
        log("EXTERNAL DATA FAILED - continuing with provided data only")
        traceback.print_exc()
        ext_df = pd.DataFrame(columns=["path", "lat", "lon", "src"])

full_df = pd.concat([train_df, ext_df], ignore_index=True)
full_df["w"] = np.where(full_df.src.values == "provided", CFG.W_PROVIDED, CFG.W_EXTERNAL)
log(f"TOTAL training rows = {len(full_df)}  "
    f"(provided={int((full_df.src=='provided').sum())}, external={int((full_df.src=='external').sum())})")

[16:44:08 | +   1.0 min] OSV-5M: 98 train zips, meta candidates=['train.csv']


train.csv:   0%|          | 0.00/2.92G [00:00<?, ?B/s]

[16:44:24 | +   1.2 min] OSV-5M meta cols -> id=id lat=latitude lon=longitude
[16:44:46 | +   1.6 min] OSV-5M meta rows: 4894684
[16:44:46 | +   1.6 min] downloading shard images/train/00.zip ...


images/train/00.zip:   0%|          | 0.00/2.52G [00:00<?, ?B/s]

[16:45:15 | +   2.1 min] extracted images/train/00.zip
[16:45:15 | +   2.1 min] downloading shard images/train/01.zip ...


...

[16:46:13 | +   3.0 min] extracted images/train/02.zip
[16:46:13 | +   3.0 min] downloading shard images/train/03.zip ...


images/train/03.zip:   0%|          | 0.00/2.51G [00:00<?, ?B/s]

[16:46:46 | +   3.6 min] extracted images/train/03.zip
[16:46:46 | +   3.6 min] downloading shard images/train/04.zip ...


images/train/04.zip:   0%|          | 0.00/2.52G [00:00<?, ?B/s]

[16:47:14 | +   4.0 min] extracted images/train/04.zip
[16:47:14 | +   4.1 min] external images on disk: 250000
[16:47:19 | +   4.1 min] external rows kept: 250000
[16:47:19 | +   4.1 min] TOTAL training rows = 269002  (provided=19002, external=250000)


In [6]:
# =====================================================================
# CELL 5B - CONTAMINATION GUARD (rule 3.2: never train on test images)
#
# Layer 1 (id match) already ran in Cells 3 and 5. It is necessary but NOT
# sufficient: the competition renamed its images, so a competition image_id
# can never equal an OSV-5M id even when the two files are the same photo.
# Layer 2 below compares IMAGE CONTENT with a perceptual hash, which is what
# actually settles the question.
# =====================================================================
RUN_PHASH_GUARD = True
HAM_THRESH      = 8

# --- layer 1, stated explicitly for the record ---
_ext_ids  = set(ext_df.get("path", pd.Series(dtype=str)).map(
                lambda p: os.path.splitext(os.path.basename(p))[0])) if len(ext_df) else set()
_id_clash = _ext_ids & test_stem_set
log(f"LAYER 1 (filename): external ids colliding with test ids = {len(_id_clash)} "
    f"(already excluded in Cell 5)")

if RUN_PHASH_GUARD and len(full_df):
    try:
        from scipy.fftpack import dct
        POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

        def _phash(img, hs=8, size=32):
            g = img.convert("L").resize((size, size), Image.BICUBIC)
            d = dct(dct(np.asarray(g, dtype=np.float64), axis=0, norm="ortho"),
                    axis=1, norm="ortho")[:hs, :hs]
            f = d.flatten()[1:]
            return np.packbits((f > np.median(f)).astype(np.uint8))

        class HashDS(Dataset):
            def __init__(self, paths): self.paths = list(paths)
            def __len__(self): return len(self.paths)
            def __getitem__(self, i):
                try:  return torch.tensor(_phash(Image.open(self.paths[i]))), 1
                except Exception: return torch.zeros(8, dtype=torch.uint8), 0

        def _hash_all(paths, tag):
            dl = DataLoader(HashDS(paths), batch_size=256, num_workers=CFG.WORKERS)
            H, O, n = [], [], 0
            for h, o in dl:
                H.append(h.numpy()); O.append(o.numpy()); n += len(h)
                if n % 25000 < 256: log(f"  [{tag}] hashed {n}/{len(paths)}")
            return np.concatenate(H), np.concatenate(O)

        def _min_hamming(A, B):
            out = np.full(len(A), 64, dtype=np.int16)
            for i in range(0, len(A), 4096):
                out[i:i+4096] = POPCOUNT[A[i:i+4096][:, None, :] ^ B[None, :, :]].sum(-1).min(1)
            return out

        log("LAYER 2 (perceptual hash): hashing test images ...")
        Ht, okt = _hash_all([p for p in TEST_PATHS if p], "test")
        Ht = Ht[okt == 1]
        log(f"  test hashes: {len(Ht)}")

        log("LAYER 2: hashing training pool (this takes a few minutes) ...")
        Hx, okx = _hash_all(full_df.path.values, "pool")
        dmin = _min_hamming(Hx, Ht)

        hit = dmin <= HAM_THRESH
        log(f"  nearest-test-image distance: p0={dmin.min()} p1={np.percentile(dmin,1):.0f} "
            f"p50={np.percentile(dmin,50):.0f} bits")
        log(f"  CONTAMINATED rows found: {int(hit.sum())} / {len(full_df)}")
        if hit.sum():
            full_df.loc[hit, ["path"]].to_csv(
                os.path.join(CFG.WORK, "contaminated_rows.csv"), index=False)
            by_src = full_df.loc[hit, "src"].value_counts().to_dict()
            log(f"  breakdown by source: {by_src}")
            full_df = full_df[~hit].reset_index(drop=True)
            log(f"  DROPPED. clean training pool = {len(full_df)}")
        else:
            log("  >>> ZERO content overlap with the test set. Pool is clean. <<<")
    except Exception as e:
        log(f"pHash guard could not run ({e}) - filename filter still applied")
        traceback.print_exc()

log(f"VERIFIED training pool: {len(full_df):,} rows, 0 test images")

[16:47:19 | +   4.1 min] LAYER 1 (filename): external ids colliding with test ids = 0 (already excluded in Cell 5)
[16:47:19 | +   4.1 min] LAYER 2 (perceptual hash): hashing test images ...
[16:47:24 | +   4.2 min]   test hashes: 500
[16:47:24 | +   4.2 min] LAYER 2: hashing training pool (this takes a few minutes) ...
[16:48:43 | +   5.5 min]   [pool] hashed 25088/269002
[16:49:48 | +   6.6 min]   [pool] hashed 50176/269002
[16:50:53 | +   7.7 min]   [pool] hashed 75008/269002
[16:51:55 | +   8.7 min]   [pool] hashed 100096/269002
[16:52:54 | +   9.7 min]   [pool] hashed 125184/269002
[16:53:47 | +  10.6 min]   [pool] hashed 150016/269002
[16:54:37 | +  11.4 min]   [pool] hashed 175104/269002
[16:55:24 | +  12.2 min]   [pool] hashed 200192/269002
[16:56:10 | +  13.0 min]   [pool] hashed 225024/269002
[16:56:56 | +  13.7 min]   [pool] hashed 250112/269002
[16:57:37 | +  14.4 min]   nearest-test-image distance: p0=4 p1=12 p50=18 bits
[16:57:37 | +  14.4 min]   CONTAMINATED rows found: 

In [7]:
# =====================================================================
# CELL 6 - Country labels from the competition's own boundary file
# =====================================================================
import shapely
from shapely.geometry import shape, Point
from shapely.strtree import STRtree
from shapely.ops import nearest_points

country_names, country_geoms = [], []
if GEOJSON:
    with open(GEOJSON, "r", encoding="utf-8") as f:
        gj = json.load(f)
    for i, feat in enumerate(gj["features"]):
        try:
            g = shape(feat["geometry"])
            if not g.is_valid:
                g = g.buffer(0)
            props = feat.get("properties", {})
            nm = (props.get("country_name") or props.get("ADMIN") or props.get("name")
                  or props.get("NAME") or props.get("iso_a2") or f"c{i}")
            country_geoms.append(g); country_names.append(str(nm))
        except Exception:
            continue
log(f"country polygons loaded: {len(country_geoms)}")

CTREE = STRtree(country_geoms) if country_geoms else None
N_COUNTRY = max(len(country_geoms), 1)

def assign_country(lats, lons):
    """Vectorised point-in-polygon. Returns -1 where the point is in no country."""
    out = np.full(len(lats), -1, dtype=np.int64)
    if CTREE is None:
        return out
    pts = shapely.points(np.asarray(lons, dtype=float), np.asarray(lats, dtype=float))
    try:
        pairs = CTREE.query(pts, predicate="intersects")
        out[pairs[0]] = pairs[1]
    except Exception:
        for i, p in enumerate(pts):
            hit = CTREE.query(p, predicate="intersects")
            if len(hit): out[i] = int(hit[0])
    return out

log("labelling training points by country ...")
full_df["country"] = assign_country(full_df.lat.values, full_df.lon.values)
in_country = int((full_df.country >= 0).sum())
log(f"points inside a country polygon: {in_country}/{len(full_df)} "
    f"({100*in_country/max(len(full_df),1):.1f}%)  [rest are coastal/ocean -> masked out]")

[16:57:38 | +  14.5 min] country polygons loaded: 298
[16:57:38 | +  14.5 min] labelling training points by country ...
[16:58:11 | +  15.0 min] points inside a country polygon: 261874/268736 (97.4%)  [rest are coastal/ocean -> masked out]


In [8]:
# =====================================================================
# CELL 7 - Geocells + SPATIAL folds (blocks, not random rows)
# =====================================================================
V = latlon_to_vec(full_df.lat.values, full_df.lon.values)

log(f"fitting fine geocells (k={CFG.N_FINE}) with MiniBatchKMeans ...")
km_fine = MiniBatchKMeans(n_clusters=CFG.N_FINE, random_state=CFG.SEED,
                          batch_size=8192, n_init=5, max_iter=300).fit(V)
full_df["fine"] = km_fine.labels_
CF = km_fine.cluster_centers_ / (np.linalg.norm(km_fine.cluster_centers_, axis=1, keepdims=True)+1e-12)
log("fine cells done")

log(f"fitting coarse geocells (k={CFG.N_COARSE}) ...")
km_coarse = MiniBatchKMeans(n_clusters=CFG.N_COARSE, random_state=CFG.SEED,
                            batch_size=8192, n_init=5, max_iter=300).fit(V)
full_df["coarse"] = km_coarse.labels_
log("coarse cells done")

# ---- haversine-smoothed soft targets, computed as ONE matmul ----
log("building smoothed target matrix (vectorised) ...")
cosm = np.clip(CF @ CF.T, -1.0, 1.0)
DM   = R_EARTH * np.arccos(cosm)
S    = np.exp(-DM / CFG.TAU_KM)
S   /= S.sum(axis=1, keepdims=True)
SOFT = torch.tensor(S, dtype=torch.float32, device=device)
CFT  = torch.tensor(CF, dtype=torch.float32, device=device)
log(f"soft target matrix {S.shape} ready")

# ---- which country does each fine cell sit in? (for constrained decoding) ----
_cf_lat, _cf_lon = vec_to_latlon(CF)
CELL_COUNTRY = assign_country(_cf_lat, _cf_lon)
np.save(os.path.join(CFG.ART, "cell_country.npy"), CELL_COUNTRY)
log(f"fine cells mapped to countries: {(CELL_COUNTRY>=0).sum()}/{len(CELL_COUNTRY)} on land")
CCT = torch.tensor(np.where(CELL_COUNTRY >= 0, CELL_COUNTRY, 0), device=device)
CC_VALID = torch.tensor((CELL_COUNTRY >= 0).astype(np.float32), device=device)

# ---- spatial blocks so validation cannot leak near-duplicate street-view frames ----
log("assigning spatial blocks for leak-free folds ...")
prov_mask = (full_df.src.values == "provided")
n_blocks = min(2000, max(6, int(prov_mask.sum() // 12)))
kb = MiniBatchKMeans(n_clusters=n_blocks, random_state=CFG.SEED, batch_size=4096,
                     n_init=5).fit(V[prov_mask])
blocks = np.full(len(full_df), -1, dtype=np.int64)
blocks[prov_mask] = kb.labels_
full_df["block"] = blocks

rng = np.random.RandomState(CFG.SEED)
perm = rng.permutation(n_blocks)
block_fold = {b: i % CFG.N_FOLDS for i, b in enumerate(perm)}
fold = np.full(len(full_df), -1, dtype=np.int64)
fold[prov_mask] = [block_fold[b] for b in blocks[prov_mask]]
full_df["fold"] = fold
log(f"{n_blocks} spatial blocks -> {CFG.N_FOLDS} folds; external data always in TRAIN")
log(full_df.groupby("fold").size().to_dict())

[16:58:12 | +  15.0 min] fitting fine geocells (k=2000) with MiniBatchKMeans ...
[16:58:35 | +  15.4 min] fine cells done
[16:58:35 | +  15.4 min] fitting coarse geocells (k=150) ...
[16:58:36 | +  15.4 min] coarse cells done
[16:58:36 | +  15.4 min] building smoothed target matrix (vectorised) ...
[16:58:36 | +  15.4 min] soft target matrix (2000, 2000) ready
[16:58:37 | +  15.4 min] fine cells mapped to countries: 1874/2000 on land
[16:58:37 | +  15.4 min] assigning spatial blocks for leak-free folds ...
[16:58:46 | +  15.6 min] 1583 spatial blocks -> 5 folds; external data always in TRAIN
[16:58:46 | +  15.6 min] {-1: 249734, 0: 3908, 1: 3996, 2: 3779, 3: 3691, 4: 3628}


In [9]:
# =====================================================================
# CELL 8 - Feature extraction (the only expensive stage; runs once)
# NOTE: no horizontal flip anywhere. Mirroring destroys driving-side,
# which is one of the strongest geolocation cues there is.
# =====================================================================
class ViewDS(Dataset):
    """Two chirality-preserving views: full-frame squash + centre square crop."""
    def __init__(self, paths): self.paths = list(paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        p = self.paths[i]; ok = 1
        try:
            im = Image.open(p).convert("RGB")
        except Exception:
            im = Image.new("RGB", (CFG.IMG, CFG.IMG), (128,128,128)); ok = 0
        w, h = im.size
        a = im.resize((CFG.IMG, CFG.IMG), Image.BICUBIC)
        s = min(w, h)
        l, t = (w-s)//2, (h-s)//2
        b = im.crop((l, t, l+s, t+s)).resize((CFG.IMG, CFG.IMG), Image.BICUBIC)
        ta = torch.from_numpy(np.asarray(a, dtype=np.uint8)).permute(2,0,1)
        tb = torch.from_numpy(np.asarray(b, dtype=np.uint8)).permute(2,0,1)
        return torch.stack([ta, tb]), ok

@torch.no_grad()
def encode(paths, tag):
    valid = [p if p is not None else "" for p in paths]
    dl = DataLoader(ViewDS(valid), batch_size=CFG.EXTRACT_BS, shuffle=False,
                    num_workers=CFG.WORKERS, pin_memory=True)
    feats, oks = [], []
    t_start = time.time(); seen = 0
    for x, ok in tqdm(dl, desc=f"encode[{tag}]", mininterval=10.0):
        B = x.shape[0]
        x = x.to(device, non_blocking=True).reshape(B*N_VIEWS, 3, CFG.IMG, CFG.IMG).half().div_(255.)
        parts = []
        for name, bk in BACKBONES.items():
            xn = (x - bk["mean"]) / bk["std"]
            o = bk["model"](pixel_values=xn)
            f = o.pooler_output if bk["kind"] == "clip" else o.last_hidden_state[:, 0]
            parts.append(f.reshape(B, N_VIEWS * bk["dim"]))
        feats.append(torch.cat(parts, 1).float().cpu().numpy().astype(np.float16))
        oks.append(ok.numpy()); seen += B
        if seen % (CFG.EXTRACT_BS * 100) < CFG.EXTRACT_BS:
            rate = seen / max(time.time()-t_start, 1e-6)
            log(f"  [{tag}] {seen}/{len(valid)} imgs | {rate:.1f} img/s | "
                f"eta {(len(valid)-seen)/max(rate,1e-6)/60:.1f} min")
    return np.concatenate(feats), np.concatenate(oks)

FEAT_TRAIN = os.path.join(CFG.WORK, "feat_train.npy")
if os.path.exists(FEAT_TRAIN):
    X = np.load(FEAT_TRAIN); log(f"loaded cached train features {X.shape}")
else:
    log(f"encoding {len(full_df)} training images (this is the long part) ...")
    X, ok_tr = encode(full_df.path.values, "train")
    np.save(FEAT_TRAIN, X)
    log(f"train features {X.shape} | unreadable images: {int((ok_tr==0).sum())}")

MU = X.astype(np.float32).mean(0); SD = X.astype(np.float32).std(0) + 1e-6
np.save(os.path.join(CFG.ART, "feat_mu.npy"), MU)
np.save(os.path.join(CFG.ART, "feat_sd.npy"), SD)
Xg = torch.tensor((X.astype(np.float32) - MU) / SD, dtype=torch.float32, device=device)
log(f"features standardised and moved to GPU: {tuple(Xg.shape)}")
del X; gc.collect()

[16:58:46 | +  15.6 min] encoding 268736 training images (this is the long part) ...


encode[train]:   0%|          | 0/5599 [00:00<?, ?it/s]

[17:00:41 | +  17.5 min]   [train] 4800/268736 imgs | 41.7 img/s | eta 105.5 min
[17:02:29 | +  19.3 min]   [train] 9600/268736 imgs | 42.9 img/s | eta 100.6 min
[17:04:19 | +  21.1 min]   [train] 14400/268736 imgs | 43.3 img/s | eta 98.0 min
[17:06:07 | +  22.9 min]   [train] 19200/268736 imgs | 43.5 img/s | eta 95.6 min
[17:07:55 | +  24.7 min]   [train] 24000/268736 imgs | 43.7 img/s | eta 93.4 min
[17:09:44 | +  26.6 min]   [train] 28800/268736 imgs | 43.7 img/s | eta 91.4 min
[17:11:33 | +  28.4 min]   [train] 33600/268736 imgs | 43.8 img/s | eta 89.5 min
[17:13:21 | +  30.2 min]   [train] 38400/268736 imgs | 43.9 img/s | eta 87.5 min
[17:15:10 | +  32.0 min]   [train] 43200/268736 imgs | 43.9 img/s | eta 85.6 min
[17:16:59 | +  33.8 min]   [train] 48000/268736 imgs | 43.9 img/s | eta 83.8 min
[17:18:47 | +  35.6 min]   [train] 52800/268736 imgs | 43.9 img/s | eta 81.9 min
[17:20:36 | +  37.4 min]   [train] 57600/268736 imgs | 44.0 img/s | eta 80.0 min
[17:22:24 | +  39.2 min]   [

100

In [10]:
# =====================================================================
# CELL 9 - Heads, losses, decoding
# =====================================================================
class GeoHead(nn.Module):
    def __init__(self, d_in, n_fine, n_coarse, n_country, hid=1024):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(d_in, hid), nn.LayerNorm(hid), nn.GELU(), nn.Dropout(0.30),
            nn.Linear(hid, hid), nn.LayerNorm(hid), nn.GELU(), nn.Dropout(0.15))
        self.fine    = nn.Linear(hid, n_fine)
        self.coarse  = nn.Linear(hid, n_coarse)
        self.country = nn.Linear(hid, n_country)
        self.delta   = nn.Linear(hid, 3)      # within-cell refinement
        self.unc     = nn.Linear(hid, 1)      # quantile regression on log error
    def forward(self, x):
        h = self.trunk(x)
        return (self.fine(h), self.coarse(h), self.country(h),
                self.delta(h), self.unc(h).squeeze(-1))

def decode_point(fine_logits, delta, topk=CFG.TOP_K, max_spread_km=600.0):
    """Top-K weighted centroid, but ONLY over cells near the top-1 cell.
    Averaging cells on opposite sides of the planet lands you in an ocean;
    the top-1 cell is always at distance 0 so the weights can never vanish."""
    p = torch.softmax(fine_logits.float(), 1)
    w, idx = torch.topk(p, topk, dim=1)
    anchor = CFT[idx[:, 0]]
    cand   = CFT[idx]
    cosang = (cand * anchor.unsqueeze(1)).sum(-1).clamp(-1+1e-9, 1-1e-9)
    w = w * ((R_EARTH * torch.acos(cosang)) <= max_spread_km).float()
    w = w / (w.sum(1, keepdim=True) + 1e-9)
    base = (cand * w.unsqueeze(-1)).sum(1)
    base = base / (base.norm(dim=1, keepdim=True) + 1e-9)
    v = base + 0.08 * torch.tanh(delta)
    return v / (v.norm(dim=1, keepdim=True) + 1e-9)

def km_between(v1, v2):
    dot = torch.clamp((v1*v2).sum(1), -1+1e-9, 1-1e-9)
    return R_EARTH * torch.acos(dot)

def country_adjust(fine_logits, country_logits, lam):
    """Re-weight cells by how much the country head believes their country.

    At ~1000 km error scale the score is dominated by the country bonus, and an
    unconstrained point head routinely lands the coordinate outside the country
    its own country head predicted - forfeiting the bonus for no reason. lam=0
    recovers the unconstrained behaviour, so lam is chosen on OOF, not assumed.
    """
    if lam <= 0:
        return fine_logits
    lp = torch.log_softmax(fine_logits.float(), 1)
    pc = torch.softmax(country_logits.float(), 1)
    cell_p = pc[:, CCT] * CC_VALID.unsqueeze(0) + 1e-6
    return lp + lam * torch.log(cell_p)

def train_fold(f, Xg, meta, epochs=CFG.EPOCHS):
    tr = np.where((meta["fold"].values != f) | (meta["src"].values == "external"))[0]
    va = np.where((meta["fold"].values == f) & (meta["src"].values == "provided"))[0]
    y_fine  = torch.tensor(meta["fine"].values.astype(np.int64),    device=device)
    y_coar  = torch.tensor(meta["coarse"].values.astype(np.int64),  device=device)
    y_ctry  = torch.tensor(meta["country"].values.astype(np.int64), device=device)
    y_vec   = torch.tensor(latlon_to_vec(meta.lat.values, meta.lon.values),
                           dtype=torch.float32, device=device)
    w_all   = torch.tensor(meta["w"].values, dtype=torch.float32, device=device)

    model = GeoHead(Xg.shape[1], CFG.N_FINE, CFG.N_COARSE, N_COUNTRY, CFG.HID).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WD)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=CFG.LR, total_steps=epochs*max(1, len(tr)//CFG.HEAD_BS), pct_start=0.25)

    for ep in range(epochs):
        model.train()
        idx = tr[torch.randperm(len(tr)).numpy()]
        tot = nb = 0
        for i in range(0, len(idx) - CFG.HEAD_BS + 1, CFG.HEAD_BS):
            b = torch.tensor(idx[i:i+CFG.HEAD_BS], device=device)
            xb, wb = Xg[b], w_all[b]
            fl, cl, ctl, dl_, ul = model(xb)

            soft = SOFT[y_fine[b]]
            l_fine = -(soft * torch.log_softmax(fl, 1)).sum(1)
            l_coar = F.cross_entropy(cl, y_coar[b], reduction="none")
            cmask  = y_ctry[b] >= 0
            l_ctry = torch.where(cmask,
                        F.cross_entropy(ctl, y_ctry[b].clamp(min=0), reduction="none"),
                        torch.zeros_like(l_coar))

            v = decode_point(fl, dl_)
            chord = (v - y_vec[b]).norm(dim=1)
            l_pt = F.huber_loss(chord, torch.zeros_like(chord), reduction="none", delta=0.05)

            with torch.no_grad():
                err = torch.log1p(km_between(v.detach(), y_vec[b]))
            diff = err - ul
            l_unc = torch.maximum(CFG.Q_UNC*diff, (CFG.Q_UNC-1)*diff)

            loss = (wb * (l_fine + 0.2*l_coar + 1.5*l_ctry + 8.0*l_pt + 0.2*l_unc)).mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step(); sched.step()
            tot += loss.item(); nb += 1

        if ep % 5 == 0 or ep == epochs-1:
            model.eval()
            with torch.no_grad():
                fl, cl, ctl, dl_, ul = model(Xg[torch.tensor(va, device=device)])
                v = decode_point(fl, dl_)
                med = km_between(v, y_vec[torch.tensor(va, device=device)]).median().item()
            log(f"    fold {f} ep {ep+1:3d}/{epochs} | loss {tot/max(nb,1):.4f} | "
                f"val median {med:8.1f} km")
    return model, tr, va

log("head definitions ready")

[18:40:57 | + 117.8 min] head definitions ready


In [11]:
# =====================================================================
# CELL 10 - Train the fold ensemble, collect out-of-fold predictions
# =====================================================================
models, OOF = [], {}
y_vec_all = torch.tensor(latlon_to_vec(full_df.lat.values, full_df.lon.values),
                         dtype=torch.float32, device=device)

for f in range(CFG.N_FOLDS):
    log(f"=== FOLD {f+1}/{CFG.N_FOLDS} ===")
    m, tr, va = train_fold(f, Xg, full_df)
    m.eval()
    with torch.no_grad():
        vi = torch.tensor(va, device=device)
        fl, cl, ctl, dl_, ul = m(Xg[vi])
        OOF[f] = dict(idx=va,
                      fine=fl.float().cpu(), ctry_logit=ctl.float().cpu(),
                      delta=dl_.float().cpu(), unc=ul.float().cpu())
    models.append(m)
    torch.save(m.state_dict(), os.path.join(CFG.ART, f"head_fold{f}.pt"))
    log(f"fold {f} saved")

oof_idx   = np.concatenate([OOF[f]["idx"] for f in range(CFG.N_FOLDS)])
oof_fine  = torch.cat([OOF[f]["fine"]       for f in range(CFG.N_FOLDS)]).to(device)
oof_ctryl = torch.cat([OOF[f]["ctry_logit"] for f in range(CFG.N_FOLDS)]).to(device)
oof_delta = torch.cat([OOF[f]["delta"]      for f in range(CFG.N_FOLDS)]).to(device)
oof_unc   = np.expm1(torch.cat([OOF[f]["unc"] for f in range(CFG.N_FOLDS)]).numpy())

true_lat  = full_df.lat.values[oof_idx]
true_lon  = full_df.lon.values[oof_idx]
true_ctry = full_df.country.values[oof_idx]

def oof_points(lam):
    with torch.no_grad():
        v = decode_point(country_adjust(oof_fine, oof_ctryl, lam), oof_delta)
    la, lo = vec_to_latlon(v.cpu().numpy())
    return la, lo

_la, _lo = oof_points(0.0)
oof_err0 = hav_km(_la, _lo, true_lat, true_lon)
oof_ctry_pred = oof_ctryl.argmax(1).cpu().numpy()
log(f"OOF median error (unconstrained): {np.median(oof_err0):8.1f} km")
log(f"OOF mean   error (unconstrained): {np.mean(oof_err0):8.1f} km")
for q in [10,25,50,75,90]:
    log(f"  p{q:<3d} error : {np.percentile(oof_err0,q):8.1f} km")
log(f"OOF country-head accuracy : {100*np.mean(oof_ctry_pred==true_ctry):.1f}%")
log(f"OOF point-in-right-country: "
    f"{100*np.mean((assign_country(_la,_lo)==true_ctry)&(true_ctry>=0)):.1f}%")

[18:40:58 | + 117.8 min] === FOLD 1/5 ===
[18:41:02 | + 117.9 min]     fold 0 ep   1/60 | loss 11.9844 | val median   1255.2 km
[18:41:18 | + 118.1 min]     fold 0 ep   6/60 | loss 6.6313 | val median    854.8 km
[18:41:36 | + 118.4 min]     fold 0 ep  11/60 | loss 6.1308 | val median    893.7 km
[18:41:53 | + 118.7 min]     fold 0 ep  16/60 | loss 5.6552 | val median    884.5 km
[18:42:09 | + 119.0 min]     fold 0 ep  21/60 | loss 5.3295 | val median    787.0 km
[18:42:25 | + 119.2 min]     fold 0 ep  26/60 | loss 5.1528 | val median    768.5 km
[18:42:41 | + 119.5 min]     fold 0 ep  31/60 | loss 5.0401 | val median    750.7 km
[18:42:57 | + 119.8 min]     fold 0 ep  36/60 | loss 4.9496 | val median    743.5 km
[18:43:14 | + 120.1 min]     fold 0 ep  41/60 | loss 4.8844 | val median    743.5 km
[18:43:30 | + 120.3 min]     fold 0 ep  46/60 | loss 4.8370 | val median    715.4 km
[18:43:47 | + 120.6 min]     fold 0 ep  51/60 | loss 4.8082 | val median    744.9 km
[18:44:03 | + 120.9 mi

In [12]:
# =====================================================================
# CELL 11 - Radius policy chosen by MAXIMIN over a family of plausible
# scoring functions. The exact metric is not published, so instead of
# fitting one guess we pick the policy with the best WORST CASE.
# =====================================================================
def proxy_score(d, r, ctry_ok, D, W=0.40, CB=0.15, RT=750.0):
    dist = np.exp(-d / D)
    cal  = W * np.exp(-r / D) * np.where(d <= r, 1.0, -1.0)
    cb   = CB * (ctry_ok & (r <= RT)).astype(float)
    return dist + cal + cb

D_GRID = [500.0, 1000.0, 1500.0, 2000.0]

from sklearn.isotonic import IsotonicRegression

def maximin(err, rad, ok):
    return min(np.median(proxy_score(err, rad, ok, D)) for D in D_GRID)

# ---- step 1: how hard should the country head pull the point? ----
log("searching lambda (country-constrained decoding) ...")
LAM_GRID = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
lam_rows = []
for lam in LAM_GRID:
    la, lo = oof_points(lam)
    e  = hav_km(la, lo, true_lat, true_lon)
    ok = (assign_country(la, lo) == true_ctry) & (true_ctry >= 0)
    # provisional radius so lambda is judged on score, not distance alone
    prov = np.clip(2.0*np.maximum(oof_unc,1.0), 30.0, CFG.R_MAX)
    lam_rows.append((maximin(e, prov, ok), lam, np.median(e), ok.mean()))
    log(f"  lam={lam:4.2f} | median err {np.median(e):7.1f} km | "
        f"in-country {100*ok.mean():5.1f}% | score {lam_rows[-1][0]:.4f}")
lam_rows.sort(reverse=True)
LAM = lam_rows[0][1]
log(f">>> chosen lambda = {LAM}")

oof_lat, oof_lon = oof_points(LAM)
oof_err   = hav_km(oof_lat, oof_lon, true_lat, true_lon)
oof_ok    = (assign_country(oof_lat, oof_lon) == true_ctry) & (true_ctry >= 0)
oof_ctry_ok_raw = oof_ok
log(f"final OOF median error {np.median(oof_err):.1f} km | "
    f"in-country {100*oof_ok.mean():.1f}%")

# ---- step 2: radius. two families, OOF decides (isotonic is NOT assumed better) ----
log("searching radius policy ...")
best_lin = None
for alpha in np.arange(0.4, 8.01, 0.2):
    for floor in [15.0, 30.0, 60.0, 100.0]:
        r = np.clip(alpha*np.maximum(oof_unc,1.0), floor, CFG.R_MAX)
        s = maximin(oof_err, r, oof_ok)
        if best_lin is None or s > best_lin[0]: best_lin = (s, float(alpha), float(floor))
log(f"  linear   : alpha={best_lin[1]:.1f} floor={best_lin[2]:.0f} -> {best_lin[0]:.4f}")

best_iso, ISO = None, None
try:
    order = np.argsort(oof_unc); bins = np.array_split(order, 20)
    xs = np.array([oof_unc[b].mean() for b in bins])
    for q in np.arange(0.35, 0.96, 0.05):
        ys  = np.array([np.percentile(oof_err[b], q*100) for b in bins])
        iso = IsotonicRegression(increasing=True, out_of_bounds="clip").fit(xs, ys)
        r   = np.clip(iso.predict(oof_unc), 15.0, CFG.R_MAX)
        s   = maximin(oof_err, r, oof_ok)
        if best_iso is None or s > best_iso[0]: best_iso, ISO = (s, float(q)), iso
    log(f"  isotonic : q={best_iso[1]:.2f} -> {best_iso[0]:.4f}")
except Exception as e:
    log(f"  isotonic skipped ({e})")

USE_ISO = (best_iso is not None) and (best_iso[0] > best_lin[0])
ALPHA, FLOOR = best_lin[1], best_lin[2]
log(f">>> radius family chosen by OOF: {'isotonic' if USE_ISO else 'linear'}")

def radius_from_unc(u):
    if USE_ISO:
        return np.clip(ISO.predict(u), 15.0, CFG.R_MAX)
    return np.clip(ALPHA*np.maximum(u,1.0), FLOOR, CFG.R_MAX)

r_oof = radius_from_unc(oof_unc)
BASE_SCORE = maximin(oof_err, r_oof, oof_ok)
log(f"  coverage: {100*np.mean(oof_err<=r_oof):.1f}% | median radius {np.median(r_oof):.0f} km")
for D in D_GRID:
    log(f"  median proxy @ D={D:6.0f}: {np.median(proxy_score(oof_err,r_oof,oof_ok,D)):.4f}")
log(f">>> STAGE-1 BASELINE SCORE = {BASE_SCORE:.4f}")

json.dump({"lam":float(LAM),"use_iso":bool(USE_ISO),"alpha":float(ALPHA),
           "floor":float(FLOOR),"r_max":float(CFG.R_MAX),"topk":CFG.TOP_K,
           "tau":CFG.TAU_KM,"n_fine":CFG.N_FINE,"base_score":float(BASE_SCORE)},
          open(os.path.join(CFG.ART,"calibration.json"),"w"))
np.save(os.path.join(CFG.ART,"fine_centroids.npy"), CF)
log("calibration + centroids saved to artifacts/")

[18:57:27 | + 134.3 min] searching lambda (country-constrained decoding) ...
[18:57:30 | + 134.3 min]   lam=0.00 | median err   687.8 km | in-country  48.6% | score 0.1690
[18:57:33 | + 134.4 min]   lam=0.25 | median err   680.6 km | in-country  51.8% | score 0.1768
[18:57:36 | + 134.4 min]   lam=0.50 | median err   684.7 km | in-country  52.0% | score 0.1729
[18:57:39 | + 134.5 min]   lam=0.75 | median err   686.6 km | in-country  52.1% | score 0.1729
[18:57:42 | + 134.5 min]   lam=1.00 | median err   690.2 km | in-country  52.1% | score 0.1719
[18:57:45 | + 134.6 min]   lam=1.50 | median err   694.6 km | in-country  52.1% | score 0.1704
[18:57:48 | + 134.6 min]   lam=2.00 | median err   695.2 km | in-country  52.1% | score 0.1702
[18:57:52 | + 134.7 min]   lam=3.00 | median err   697.8 km | in-country  52.1% | score 0.1694
[18:57:52 | + 134.7 min] >>> chosen lambda = 0.25
[18:57:55 | + 134.7 min] final OOF median error 680.6 km | in-country 51.8%
[18:57:55 | + 134.7 min] searching ra

In [13]:
# =====================================================================
# CELL 12 - Test inference (rerun this + Cell 13 for hidden Test Set 2)
# Point TEST_DIR_OVERRIDE at the new folder, rerun, done.
# =====================================================================
TEST_DIR_OVERRIDE = None   # <-- for round 2, set this to the new image folder

if TEST_DIR_OVERRIDE:
    sub_template = pd.read_csv(SAMPLE_SUB)   # replace with round-2 sample sub if different
    tp = {os.path.splitext(os.path.basename(p))[0]: p
          for p in glob.glob(os.path.join(TEST_DIR_OVERRIDE, "**", "*.*"), recursive=True)}
    TEST_PATHS = [tp.get(os.path.splitext(str(t))[0]) for t in sub_template[SUB_ID].astype(str)]

log(f"encoding {len(TEST_PATHS)} test images ...")
Xt, ok_te = encode(TEST_PATHS, "test")
log(f"test features {Xt.shape} | unreadable: {int((ok_te==0).sum())}")
Xtg = torch.tensor((Xt.astype(np.float32) - MU) / SD, dtype=torch.float32, device=device)

probs_sum = torch.zeros(len(Xtg), CFG.N_FINE, device=device)
delta_sum = torch.zeros(len(Xtg), 3, device=device)
ctry_sum  = torch.zeros(len(Xtg), N_COUNTRY, device=device)
unc_sum   = torch.zeros(len(Xtg), device=device)

with torch.no_grad():
    for f, m in enumerate(models):
        m.eval()
        for i in range(0, len(Xtg), 4096):
            xb = Xtg[i:i+4096]
            fl, cl, ctl, dl_, ul = m(xb)
            probs_sum[i:i+4096] += torch.softmax(fl.float(), 1)
            delta_sum[i:i+4096] += dl_.float()
            ctry_sum[i:i+4096]  += torch.softmax(ctl.float(), 1)
            unc_sum[i:i+4096]   += ul.float()
        log(f"  ensembled fold {f+1}/{len(models)}")

probs = probs_sum / len(models)
delta = delta_sum / len(models)
ctryp = ctry_sum  / len(models)
uncp  = unc_sum   / len(models)

v = decode_point(country_adjust(torch.log(probs + 1e-12),
                                torch.log(ctryp + 1e-12), LAM), delta)
pred_lat, pred_lon = vec_to_latlon(v.cpu().numpy())
pred_rad = radius_from_unc(np.expm1(uncp.cpu().numpy()))
log("raw predictions decoded")

# ---- ocean rescue: any point in no country gets pulled to nearest land ----
where = assign_country(pred_lat, pred_lon)
ocean = np.where(where < 0)[0]
log(f"points landing outside every country polygon: {len(ocean)}")
for j in ocean:
    try:
        p = Point(float(pred_lon[j]), float(pred_lat[j]))
        gi = CTREE.nearest(p)
        gi = int(gi if np.isscalar(gi) else np.asarray(gi).ravel()[0])
        q, _ = nearest_points(country_geoms[gi], p)
        pred_lat[j], pred_lon[j] = q.y, q.x
    except Exception:
        pass
log("ocean predictions snapped to nearest land")

pred_lat = np.clip(pred_lat, -90, 90)
pred_lon = ((np.asarray(pred_lon) + 180) % 360) - 180
bad = ~np.isfinite(pred_lat) | ~np.isfinite(pred_lon) | ~np.isfinite(pred_rad)
if bad.any():
    log(f"WARNING: {int(bad.sum())} non-finite predictions replaced with safe defaults")
    pred_lat[bad], pred_lon[bad], pred_rad[bad] = 0.0, 0.0, 2000.0

[18:57:55 | + 134.7 min] encoding 500 test images ...


encode[test]:   0%|          | 0/11 [00:00<?, ?it/s]

[18:58:09 | + 135.0 min] test features (500, 2048) | unreadable: 0
[18:58:09 | + 135.0 min]   ensembled fold 1/5
[18:58:09 | + 135.0 min]   ensembled fold 2/5
[18:58:09 | + 135.0 min]   ensembled fold 3/5
[18:58:09 | + 135.0 min]   ensembled fold 4/5
[18:58:09 | + 135.0 min]   ensembled fold 5/5
[18:58:09 | + 135.0 min] raw predictions decoded
[18:58:09 | + 135.0 min] points landing outside every country polygon: 46
[18:58:09 | + 135.0 min] ocean predictions snapped to nearest land


In [14]:
# =====================================================================
# CELL 13 - Write submission in EXACTLY the sample file's schema
# =====================================================================
sub = sub_template.copy()
sub[SUB_LAT] = pred_lat
sub[SUB_LON] = pred_lon
sub[SUB_RAD] = pred_rad
sub = sub[list(sub_template.columns)]           # preserve column order

assert len(sub) == len(sub_template), "row count changed!"
assert sub.isna().sum().sum() == 0, "NaNs in submission!"

OUT = os.path.join(CFG.WORK, "submission.csv")
sub.to_csv(OUT, index=False)
log(f"submission written -> {OUT}")
print(sub.head(10).to_string(), flush=True)
log(f"lat range   {sub[SUB_LAT].min():.3f} .. {sub[SUB_LAT].max():.3f}")
log(f"lon range   {sub[SUB_LON].min():.3f} .. {sub[SUB_LON].max():.3f}")
log(f"radius med  {sub[SUB_RAD].median():.0f} km  "
    f"(min {sub[SUB_RAD].min():.0f}, max {sub[SUB_RAD].max():.0f})")

[18:58:09 | + 135.0 min] submission written -> /kaggle/working/submission.csv
               image_id   pred_lat    pred_lon  pred_radius_km
0  34f65e00cc3df67d.jpg  14.730512  -16.175044      649.124146
1  14fabfc3e0d31fc7.jpg  -7.425492   25.785817      616.101135
2  88a891ffc6beabac.jpg  -0.948283  -78.799277      468.186035
3  f8bec9bfff55065d.jpg  41.423199  -84.103475      712.039612
4  d16ea14697a3c421.jpg   7.912156  100.183819      731.900452
5  3b2cef528f285b83.jpg  20.771801  -88.755274      665.230225
6  af38392885b9f454.jpg  51.149708    7.035578      512.510620
7  1574541bd6040c32.jpg -24.621309   29.467462      624.551208
8  5ac5f6f7964b4624.jpg  38.516404   55.814225      612.833679
9  f7b90fe9a5323a50.jpg  13.928307  101.915313      679.635437
[18:58:09 | + 135.0 min] lat range   -44.011 .. 69.006
[18:58:09 | + 135.0 min] lon range   -156.982 .. 175.256
[18:58:09 | + 135.0 min] radius med  638 km  (min 273, max 1221)


In [15]:
# =====================================================================
# CELL 14B - STAGE 2: fine-tune the backbone.
# The OSV-5M ablation found unfreezing the last transformer blocks recovers
# most of full fine-tuning at a fraction of the cost, and beats LoRA. A frozen
# probe cannot adapt its features to the task; this is the one lever left that
# changes the representation itself.
#
# SAFETY: stage 1 has already written submission.csv. This stage only
# overwrites it if it beats BASE_SCORE on the same held-out data.
# =====================================================================
if not CFG.RUN_FINETUNE:
    log("stage 2 disabled")
else:
  try:
    import copy
    from torch.amp import autocast, GradScaler

    for bk in BACKBONES.values():
        bk["model"].cpu()
    del clip_m
    gc.collect(); torch.cuda.empty_cache()

    ft = CLIPVisionModel.from_pretrained(CLIP_LOCAL).to(device)
    for p in ft.parameters(): p.requires_grad = False
    layers = ft.vision_model.encoder.layers
    for l in layers[-CFG.FT_UNFREEZE_BLOCKS:]:
        for p in l.parameters(): p.requires_grad = True
    for p in ft.vision_model.post_layernorm.parameters(): p.requires_grad = True
    n_tr = sum(p.numel() for p in ft.parameters() if p.requires_grad)
    log(f"stage 2: unfroze last {CFG.FT_UNFREEZE_BLOCKS} blocks ({n_tr/1e6:.1f}M params)")

    head = GeoHead(BACKBONES["clip"]["dim"]*N_VIEWS, CFG.N_FINE,
                   CFG.N_COARSE, N_COUNTRY, CFG.HID).to(device)
    head.load_state_dict(torch.load(os.path.join(CFG.ART,"head_fold0.pt"),
                                    map_location=device))
    log("stage 2: head warm-started from fold 0")

    va_ft = np.where((full_df["fold"].values == 0) &
                     (full_df["src"].values == "provided"))[0]
    tr_ft = np.where((full_df["fold"].values != 0) |
                     (full_df["src"].values == "external"))[0]
    if len(tr_ft) > CFG.FT_MAX_IMAGES:
        tr_ft = np.random.RandomState(CFG.SEED).choice(tr_ft, CFG.FT_MAX_IMAGES, False)
    log(f"stage 2: train {len(tr_ft)} / val {len(va_ft)}")

    MEAN_T = BACKBONES["clip"]["mean"].float()
    STD_T  = BACKBONES["clip"]["std"].float()

    class FTDS(Dataset):
        """Augmented views. NO horizontal flip - it destroys driving side."""
        def __init__(self, idxs, train):
            self.p = full_df.path.values[idxs]; self.i = idxs; self.train = train
        def __len__(self): return len(self.i)
        def __getitem__(self, k):
            try: im = Image.open(self.p[k]).convert("RGB")
            except Exception: im = Image.new("RGB",(CFG.IMG,CFG.IMG),(128,128,128))
            w,h = im.size
            if self.train:
                sc = np.random.uniform(0.75, 1.0)
                cw, ch = int(w*sc), int(h*sc)
                x0 = np.random.randint(0, max(1, w-cw+1))
                y0 = np.random.randint(0, max(1, h-ch+1))
                im = im.crop((x0, y0, x0+cw, y0+ch))
                w, h = im.size
            a = im.resize((CFG.IMG,CFG.IMG), Image.BICUBIC)
            s = min(w,h); l,t = (w-s)//2,(h-s)//2
            b = im.crop((l,t,l+s,t+s)).resize((CFG.IMG,CFG.IMG), Image.BICUBIC)
            ta = torch.from_numpy(np.asarray(a,dtype=np.uint8)).permute(2,0,1)
            tb = torch.from_numpy(np.asarray(b,dtype=np.uint8)).permute(2,0,1)
            return torch.stack([ta,tb]), int(self.i[k])

    def ft_forward(x):
        B = x.shape[0]
        x = x.to(device, non_blocking=True).reshape(B*N_VIEWS,3,CFG.IMG,CFG.IMG)
        x = x.float().div_(255.).sub_(MEAN_T).div_(STD_T)
        f = ft(pixel_values=x).pooler_output
        return head(f.reshape(B, N_VIEWS*BACKBONES["clip"]["dim"]))

    y_fine_t = torch.tensor(full_df["fine"].values.astype(np.int64), device=device)
    y_coar_t = torch.tensor(full_df["coarse"].values.astype(np.int64), device=device)
    y_ctry_t = torch.tensor(full_df["country"].values.astype(np.int64), device=device)
    y_vec_t  = torch.tensor(latlon_to_vec(full_df.lat.values, full_df.lon.values),
                            dtype=torch.float32, device=device)
    w_t      = torch.tensor(full_df["w"].values, dtype=torch.float32, device=device)

    opt = torch.optim.AdamW([
        {"params":[p for p in ft.parameters() if p.requires_grad], "lr":CFG.FT_LR},
        {"params":head.parameters(), "lr":CFG.FT_HEAD_LR}], weight_decay=1e-4)
    dl_tr = DataLoader(FTDS(tr_ft,True), batch_size=CFG.FT_BS, shuffle=True,
                       num_workers=CFG.WORKERS, pin_memory=True, drop_last=True)
    steps = CFG.FT_EPOCHS*len(dl_tr)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[CFG.FT_LR, CFG.FT_HEAD_LR], total_steps=steps, pct_start=0.15)
    scaler = GradScaler("cuda")
    t_ft = time.time(); stop = False

    for ep in range(CFG.FT_EPOCHS):
        if stop: break
        ft.train(); head.train(); run = 0.0; nb = 0
        for x, bi in dl_tr:
            b = bi.to(device)
            with autocast("cuda", dtype=torch.float16):
                fl, cl, ctl, dl_, ul = ft_forward(x)
            fl, cl, ctl, dl_, ul = fl.float(), cl.float(), ctl.float(), dl_.float(), ul.float()
            soft   = SOFT[y_fine_t[b]]
            l_fine = -(soft*torch.log_softmax(fl,1)).sum(1)
            l_coar = F.cross_entropy(cl, y_coar_t[b], reduction="none")
            l_ctry = torch.where(y_ctry_t[b] >= 0,
                        F.cross_entropy(ctl, y_ctry_t[b].clamp(min=0), reduction="none"),
                        torch.zeros_like(l_coar))
            v      = decode_point(fl, dl_)
            chord  = (v - y_vec_t[b]).norm(dim=1)
            l_pt   = F.huber_loss(chord, torch.zeros_like(chord),
                                  reduction="none", delta=0.05)
            with torch.no_grad():
                e = torch.log1p(km_between(v.detach(), y_vec_t[b]))
            diff  = e - ul
            l_unc = torch.maximum(CFG.Q_UNC*diff, (CFG.Q_UNC-1)*diff)
            loss = (w_t[b]*(l_fine + 0.2*l_coar + 1.5*l_ctry
                            + 8.0*l_pt + 0.2*l_unc)).mean()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(
                [p for p in list(ft.parameters())+list(head.parameters())
                 if p.requires_grad], 2.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item(); nb += 1
            if nb % 200 == 0:
                el = (time.time()-t_ft)/60
                log(f"  ft ep{ep+1} step {nb}/{len(dl_tr)} | loss {run/nb:.4f} | "
                    f"{el:.1f} min elapsed")
                if el > CFG.FT_MAX_MINUTES:
                    log("  time guard hit - stopping fine-tune early"); stop = True; break
        log(f"  ft epoch {ep+1}/{CFG.FT_EPOCHS} done | loss {run/max(nb,1):.4f}")

    @torch.no_grad()
    def ft_predict(idxs):
        ft.eval(); head.eval()
        dl = DataLoader(FTDS(idxs, False), batch_size=64, shuffle=False,
                        num_workers=CFG.WORKERS, pin_memory=True)
        F_,C_,D_,U_ = [],[],[],[]
        for x,_ in dl:
            with autocast("cuda", dtype=torch.float16):
                fl,cl,ctl,dl2,ul = ft_forward(x)
            F_.append(fl.float().cpu()); C_.append(ctl.float().cpu())
            D_.append(dl2.float().cpu()); U_.append(ul.float().cpu())
        return (torch.cat(F_),torch.cat(C_),torch.cat(D_),torch.cat(U_))

    log("stage 2: predicting held-out fold ...")
    fF,fC,fD,fU = ft_predict(va_ft)
    fF,fC,fD = fF.to(device),fC.to(device),fD.to(device)
    tl, to_ = full_df.lat.values[va_ft], full_df.lon.values[va_ft]
    tc = full_df.country.values[va_ft]

    best2 = None
    for lam in LAM_GRID:
        with torch.no_grad():
            v2 = decode_point(country_adjust(fF, fC, lam), fD)
        la2, lo2 = vec_to_latlon(v2.cpu().numpy())
        e2  = hav_km(la2, lo2, tl, to_)
        ok2 = (assign_country(la2, lo2) == tc) & (tc >= 0)
        u2  = np.expm1(fU.numpy())
        for alpha in np.arange(0.4, 8.01, 0.2):
            for floor in [15.0, 30.0, 60.0, 100.0]:
                r2 = np.clip(alpha*np.maximum(u2,1.0), floor, CFG.R_MAX)
                s2 = maximin(e2, r2, ok2)
                if best2 is None or s2 > best2[0]:
                    best2 = (s2, lam, float(alpha), float(floor),
                             float(np.median(e2)), float(ok2.mean()))
    log(f"stage 2 held-out: score {best2[0]:.4f} (stage 1 = {BASE_SCORE:.4f}) | "
        f"median err {best2[4]:.1f} km | in-country {100*best2[5]:.1f}%")

    if best2[0] > BASE_SCORE:
        log(">>> stage 2 WINS - regenerating submission")
        _, LAM2, A2, FL2, _, _ = best2

        class TestFTDS(Dataset):
            def __init__(self, paths): self.p = [p if p else "" for p in paths]
            def __len__(self): return len(self.p)
            def __getitem__(self, k):
                try: im = Image.open(self.p[k]).convert("RGB")
                except Exception: im = Image.new("RGB",(CFG.IMG,CFG.IMG),(128,128,128))
                w,h = im.size; s = min(w,h); l,t = (w-s)//2,(h-s)//2
                a = im.resize((CFG.IMG,CFG.IMG), Image.BICUBIC)
                b = im.crop((l,t,l+s,t+s)).resize((CFG.IMG,CFG.IMG), Image.BICUBIC)
                return torch.stack([
                    torch.from_numpy(np.asarray(a,dtype=np.uint8)).permute(2,0,1),
                    torch.from_numpy(np.asarray(b,dtype=np.uint8)).permute(2,0,1)])

        ft.eval(); head.eval()
        dlT = DataLoader(TestFTDS(TEST_PATHS), batch_size=64, shuffle=False,
                         num_workers=CFG.WORKERS)
        Fs,Cs,Ds,Us = [],[],[],[]
        with torch.no_grad():
            for x in dlT:
                with autocast("cuda", dtype=torch.float16):
                    fl,cl,ctl,dl2,ul = ft_forward(x)
                Fs.append(fl.float()); Cs.append(ctl.float())
                Ds.append(dl2.float()); Us.append(ul.float().cpu())
        tF,tC,tD = torch.cat(Fs), torch.cat(Cs), torch.cat(Ds)
        tU = torch.cat(Us).numpy()
        with torch.no_grad():
            v3 = decode_point(country_adjust(tF, tC, LAM2), tD)
        p_lat, p_lon = vec_to_latlon(v3.cpu().numpy())
        p_rad = np.clip(A2*np.maximum(np.expm1(tU),1.0), FL2, CFG.R_MAX)
        for j in np.where(assign_country(p_lat, p_lon) < 0)[0]:
            try:
                pt = Point(float(p_lon[j]), float(p_lat[j]))
                gi = CTREE.nearest(pt)
                gi = int(gi if np.isscalar(gi) else np.asarray(gi).ravel()[0])
                q,_ = nearest_points(country_geoms[gi], pt)
                p_lat[j], p_lon[j] = q.y, q.x
            except Exception: pass
        p_lat = np.clip(p_lat,-90,90); p_lon = ((p_lon+180)%360)-180
        bad = ~np.isfinite(p_lat)|~np.isfinite(p_lon)|~np.isfinite(p_rad)
        p_lat[bad], p_lon[bad], p_rad[bad] = 0.0, 0.0, 2000.0
        sub2 = sub_template.copy()
        sub2[SUB_LAT], sub2[SUB_LON], sub2[SUB_RAD] = p_lat, p_lon, p_rad
        sub2 = sub2[list(sub_template.columns)]
        assert sub2.isna().sum().sum() == 0
        sub2.to_csv(OUT, index=False)
        log(f"STAGE 2 SUBMISSION WRITTEN -> {OUT}")
        log(f"  radius median {sub2[SUB_RAD].median():.0f} km")
    else:
        log(">>> stage 2 did NOT beat stage 1 - keeping the stage-1 submission")
    torch.save({"backbone":ft.state_dict(),"head":head.state_dict()},
               os.path.join(CFG.ART,"finetuned.pt"))
    log("stage 2 weights saved")
  except Exception as e:
    log(f"STAGE 2 FAILED - stage 1 submission is untouched: {e}")
    traceback.print_exc()

[18:58:35 | + 135.4 min] stage 2: unfroze last 4 blocks (50.4M params)
[18:58:36 | + 135.4 min] stage 2: head warm-started from fold 0
[18:58:36 | + 135.4 min] stage 2: train 150000 / val 3908
[19:02:13 | + 139.0 min]   ft ep1 step 200/4687 | loss 8.3588 | 3.6 min elapsed
[19:05:47 | + 142.6 min]   ft ep1 step 400/4687 | loss 7.6315 | 7.2 min elapsed
[19:09:22 | + 146.2 min]   ft ep1 step 600/4687 | loss 7.2668 | 10.8 min elapsed
[19:12:56 | + 149.8 min]   ft ep1 step 800/4687 | loss 6.9870 | 14.3 min elapsed
[19:16:31 | + 153.3 min]   ft ep1 step 1000/4687 | loss 6.8507 | 17.9 min elapsed
[19:20:05 | + 156.9 min]   ft ep1 step 1200/4687 | loss 6.7385 | 21.5 min elapsed
[19:23:39 | + 160.5 min]   ft ep1 step 1400/4687 | loss 6.7046 | 25.1 min elapsed
[19:27:14 | + 164.1 min]   ft ep1 step 1600/4687 | loss 6.6763 | 28.6 min elapsed
[19:30:48 | + 167.6 min]   ft ep1 step 1800/4687 | loss 6.6914 | 32.2 min elapsed
[19:34:23 | + 171.2 min]   ft ep1 step 2000/4687 | loss 6.7006 | 35.8 min e

In [16]:
# =====================================================================
# CELL 14 - Rule 4.3 proof: reload every weight with the hub DISABLED
# =====================================================================
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
try:
    _c = CLIPVisionModel.from_pretrained(CLIP_LOCAL)
    log("OFFLINE CHECK: CLIP reloaded from local path with hub disabled  [PASS]")
    del _c
    if CFG.USE_DINO:
        _d = AutoModel.from_pretrained(DINO_LOCAL)
        log("OFFLINE CHECK: DINOv2 reloaded from local path              [PASS]")
        del _d
    _h = GeoHead(FEAT_DIM, CFG.N_FINE, CFG.N_COARSE, N_COUNTRY, CFG.HID)
    _h.load_state_dict(torch.load(os.path.join(CFG.ART, "head_fold0.pt"),
                                  map_location="cpu"))
    log("OFFLINE CHECK: trained head reloaded from artifacts         [PASS]")
    del _h
    log(">>> INFERENCE PATH IS FULLY OFFLINE-CAPABLE (rule 4.3 satisfied) <<<")
except Exception as e:
    log(f"OFFLINE CHECK FAILED: {e}")
    traceback.print_exc()
finally:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)
gc.collect(); torch.cuda.empty_cache()

log("artifacts directory contents:")
for p in sorted(glob.glob(os.path.join(CFG.ART, "**", "*"), recursive=True)):
    if os.path.isfile(p):
        log(f"  {os.path.relpath(p, CFG.ART):45s} {os.path.getsize(p)/1e6:8.1f} MB")

log("=" * 70)
log("RUN COMPLETE")
log(f"OOF median error   : {np.median(oof_err):.1f} km")
log(f"OOF country acc    : {100*np.mean(oof_ctry_ok_raw):.1f}%")
log(f"radius policy      : {ALPHA} x predicted_error, floor {FLOOR} km")
log(f"submission         : {OUT}")
log("=" * 70)

[22:23:41 | + 340.5 min] OFFLINE CHECK: CLIP reloaded from local path with hub disabled  [PASS]
[22:23:41 | + 340.5 min] OFFLINE CHECK: trained head reloaded from artifacts         [PASS]
[22:23:41 | + 340.5 min] >>> INFERENCE PATH IS FULLY OFFLINE-CAPABLE (rule 4.3 satisfied) <<<
[22:23:42 | + 340.5 min] artifacts directory contents:
[22:23:42 | + 340.5 min]   calibration.json                                   0.0 MB
[22:23:42 | + 340.5 min]   cell_country.npy                                   0.0 MB
[22:23:42 | + 340.5 min]   clip_vit_l14/config.json                           0.0 MB
[22:23:42 | + 340.5 min]   clip_vit_l14/model.safetensors                  1212.8 MB
[22:23:42 | + 340.5 min]   feat_mu.npy                                        0.0 MB
[22:23:42 | + 340.5 min]   feat_sd.npy                                        0.0 MB
[22:23:42 | + 340.5 min]   fine_centroids.npy                                 0.0 MB
[22:23:42 | + 340.5 min]   finetuned.pt                             